# 3A · DataFrame Fundamentals
### Financial Analytics — Module 3

In Module 2 you valued a portfolio with loops. pandas does the same job on **millions of rows** with less code — because someone already wrote the loops for you, in fast compiled form.

**The two objects to know:**
- **Series** — one labelled column of data
- **DataFrame** — a table of Series sharing an index (think: a smart spreadsheet)

In [ ]:
import pandas as pd

# Load the course dataset. This URL pattern works in Colab AND locally.
import os
BASE = "data/" if os.path.exists("data") else "https://raw.githubusercontent.com/vivekhashtag/financial-analytics-course/main/data/"
prices = pd.read_csv(BASE + "nifty50_prices.csv", parse_dates=["date"])

prices.head()     # first 5 rows - ALWAYS look at your data first

`parse_dates=["date"]` tells pandas that column holds dates, not text. Forget it, and every date operation later breaks quietly.

In [ ]:
# The three commands you run on EVERY new dataset, in this order:
print(prices.shape)        # (rows, columns)
print()
prices.info()              # column types + non-null counts <- spot missing data instantly

**Read that `info()` output carefully.** `volume` shows fewer non-null values than the other columns — pandas just told you, for free, that Module 1's planted defect (volume missing for 2021) is real. This is why `info()` comes before any analysis.

In [ ]:
prices.describe()          # summary statistics for every numeric column

**Spot anything odd in `describe()`?** Look at the `max` of the `high` column versus the max of `close`. One of these numbers is 10x the other — that's the fat-finger spike from the dataset registry. `describe()` found it in one line.

---
## Selecting data

Three patterns cover 90% of daily work.

In [ ]:
# 1. Select a column -> you get a Series
closes = prices["close"]
print(type(closes).__name__)
closes.head(3)

In [ ]:
# 2. Filter rows with a condition (a "boolean mask")
recent = prices[prices["date"] >= "2025-01-01"]
print(len(recent), "rows in 2025")

# Combine conditions: & for AND, | for OR - parentheses are REQUIRED
big_up_days = prices[(prices["close"] > prices["open"]) & (prices["date"] >= "2025-01-01")]
print(len(big_up_days), "up-days in 2025")

In [ ]:
# 3. loc: rows and columns by label, together
prices.loc[prices["date"] == "2024-08-13", ["date", "high", "close"]]

There it is — the fat-finger row, isolated in one line. In Module 2 that would have been a loop with an `if` inside. Same logic, industrial strength.

### ✏️ Exercise 1
Filter to all rows where `close` is above 17,000 **and** the date is in 2025. How many are there?

In [ ]:
# your code here


---
## Creating new columns

New columns are assignments. pandas applies the arithmetic to **every row at once** — no loop. This is called *vectorisation*.

In [ ]:
prices["range"] = prices["high"] - prices["low"]              # intraday range in points
prices["return_pct"] = prices["close"].pct_change() * 100     # day-on-day % change

prices[["date", "close", "range", "return_pct"]].head()

`pct_change()` compares each row with the previous one — the up-days loop you wrote by hand in Module 2, now built in. The first row is `NaN` because there's no previous day. That's correct, not broken.

In [ ]:
# Sorting and ranking
worst_days = prices.sort_values("return_pct").head(5)
worst_days[["date", "close", "return_pct"]]

### ✏️ Exercise 2
Find the 5 **best** days by `return_pct`. (One argument changes.)

In [ ]:
# your code here


---
## Dates as a superpower

Once a column is a real datetime, pandas gives you calendar arithmetic free.

In [ ]:
prices["year"] = prices["date"].dt.year
prices["weekday"] = prices["date"].dt.day_name()

# Average return by weekday - a classic (and classically overhyped) market question
prices.groupby("weekday")["return_pct"].mean().round(3)

*(A taste of `groupby` — notebook 3C goes deep on it.)*

### ✏️ Exercise 3
Compute the average `range` per **year**. Which year was choppiest? Does that match the regime change described in the dataset registry?

In [ ]:
# your code here


---
## Recap

| Task | pandas |
|---|---|
| Load a CSV | `pd.read_csv(path, parse_dates=[...])` |
| First look | `.head()` `.shape` `.info()` `.describe()` |
| Pick a column | `df["col"]` |
| Filter rows | `df[df["col"] > x]` — `&` and `|` with parentheses |
| Rows + cols by label | `df.loc[mask, ["a","b"]]` |
| New column | `df["new"] = ...` (vectorised, no loop) |
| Day-on-day change | `df["col"].pct_change()` |
| Sort | `df.sort_values("col")` |

**Next:** 3B — cleaning the horror file. Everything Module 1 made you *find* by eye, you now *fix* with code.

---
*AI disclosure: ______*